<a href="https://colab.research.google.com/github/huseyincenik/john_snow_labs/blob/main/generating_conll_files_from_pretrained_models/notebooks/data_prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preparation - NER Pipeline and CoNLL Generation

This notebook loads healthcare datasets, runs NER pipeline, extracts entities, and saves them in CoNLL format.

**Google Drive Integration:**
- All files are saved to Google Drive
- Files are read from Google Drive
- All code is embedded in this notebook (no external Python files required)

## Steps:
1. **Google Drive Connection** - Mount Google Drive
2. **Setup & License** - Spark NLP Healthcare license and environment setup
3. **Dataset Loading** - Load and prepare healthcare datasets
4. **NER Pipeline** - Run NER pipeline with pre-trained models
5. **Entity Extraction** - Extract and merge entities (Priority: Posology > DeID > Clinical)
6. **CoNLL Generation** - Convert entities to CoNLL format

**Outputs:**
- `data/processed/text_data.csv` - Prepared text data
- `data/processed/entities.csv` - Extracted entities
- `data/conll/conll2003_text_file.conll` - CoNLL format training data


## 1. Google Drive Connection


In [ ]:
# Mount Google Drive
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

# Set project folder in Google Drive
PROJECT_FOLDER = '/content/drive/MyDrive/john_snow_labs_ner'
os.makedirs(PROJECT_FOLDER, exist_ok=True)

# Change working directory
os.chdir(PROJECT_FOLDER)

# Create folder structure
for folder in ['data/raw', 'data/processed', 'data/conll', 'cache_pretrained']:
    os.makedirs(folder, exist_ok=True)

print(f"✅ Google Drive mounted")
print(f"✅ Project folder: {PROJECT_FOLDER}")
print(f"✅ Current directory: {os.getcwd()}")


Mounted at /content/drive
✅ Google Drive mounted
✅ Project folder: /content/drive/MyDrive/john_snow_labs_ner
✅ Current directory: /content/drive/MyDrive/john_snow_labs_ner


## 2. Setup & License Configuration


In [ ]:
import json
import os

# Load license keys from Google Drive
license_path = f'{PROJECT_FOLDER}/spark_jsl.json'
if not os.path.exists(license_path):
    print("❌ License file not found!")
    print("Please upload spark_jsl.json to Google Drive at the project folder")
    print("You can upload it manually or use the following code:")
    print("from google.colab import files")
    print("uploaded = files.upload()")
    raise FileNotFoundError(f"License file not found at {license_path}")

with open(license_path) as f:
    license_keys = json.load(f)

# Set license keys as environment variables
locals().update(license_keys)
os.environ.update(license_keys)

print("✅ License keys loaded")
print(f"JSL Version: {license_keys.get('JSL_VERSION', 'N/A')}")
print(f"Public Version: {license_keys.get('PUBLIC_VERSION', 'N/A')}")


✅ License keys loaded
JSL Version: 6.1.1
Public Version: 6.1.3


In [ ]:
# Install Java (required for Spark)
import subprocess

try:
    java_version = subprocess.check_output(['java', '-version'], stderr=subprocess.STDOUT, text=True)
    print(f"✅ Java is already installed: {java_version.split(chr(10))[0]}")

    if 'JAVA_HOME' not in os.environ:
        java_paths = [
            "/usr/lib/jvm/java-11-openjdk-amd64",
            "/usr/lib/jvm/java-8-openjdk-amd64",
            "/usr/lib/jvm/default-java"
        ]
        for path in java_paths:
            if os.path.exists(path):
                os.environ["JAVA_HOME"] = path
                print(f"✅ Set JAVA_HOME to: {path}")
                break
except Exception as e:
    print(f"Java check failed: {e}")
    print("Installing Java 11...")
    os.system('apt-get update -qq > /dev/null 2>&1')
    os.system('apt-get -y install -qq openjdk-11-jdk > /dev/null 2>&1')
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
    print("✅ Java 11 installation attempted")

# Check GPU availability
gpu_check = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
has_gpu = gpu_check.returncode == 0

if has_gpu:
    print("🚀 GPU detected! Installing PyTorch with CUDA support...")
    %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
else:
    print("Installing PyTorch (CPU version)...")
    %pip install -q torch torchvision torchaudio

# Install PySpark and Spark NLP
%pip install --upgrade -q pyspark==3.4.1 spark-nlp==$PUBLIC_VERSION

# Install Spark NLP Healthcare
%pip install --upgrade -q spark-nlp-jsl==$JSL_VERSION --extra-index-url https://pypi.johnsnowlabs.com/$SECRET

# Install additional dependencies
%pip install -q pandas numpy tqdm requests

print("✅ All libraries installed successfully!")
if has_gpu:
    print("✅ GPU-accelerated PyTorch installed")


✅ Java is already installed: openjdk version "17.0.16" 2025-07-15
🚀 GPU detected! Installing PyTorch with CUDA support...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 MB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 737.0/737.0 kB 55.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 0.8.3 requires pyspark[connect]~=3.5.1, but you have pyspark 3.4.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.7/564.7 kB 9.6 MB/s eta 0:00:00
✅ All libraries installed successfully!
✅ GPU-accelerated PyTorch installed


In [ ]:
import sparknlp
import sparknlp_jsl
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetector, Tokenizer
from sparknlp_jsl.annotator import MedicalNerModel, NerConverter
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
try:
    import torch
    gpu_available = torch.cuda.is_available()
    if gpu_available:
        gpu_name = torch.cuda.get_device_name(0)
        print(f"🚀 GPU Detected: {gpu_name}")
    else:
        print("⚠️  No GPU detected. Using CPU mode.")
except ImportError:
    print("⚠️  PyTorch not available. GPU check skipped.")
    gpu_available = False

# Spark configuration
params = {
    "spark.driver.memory": "8G",
    "spark.kryoserializer.buffer.max": "2000M",
    "spark.driver.maxResultSize": "2000M",
    "spark.sql.execution.arrow.pyspark.enabled": "true",
    "spark.serializer": "org.apache.spark.serializer.KryoSerializer"
}

if gpu_available:
    params.update({
        "spark.jsl.settings.pretrained.cache_folder": f"{PROJECT_FOLDER}/cache_pretrained",
        "spark.jsl.settings.storage.cluster_tmp_dir": f"{PROJECT_FOLDER}/cache_pretrained",
        "spark.jsl.settings.annotator.gpu": "true"
    })
    print("✅ GPU acceleration enabled in Spark configuration")

# Start Spark session
try:
    print("Starting Spark session...")
    spark = sparknlp_jsl.start(license_keys['SECRET'], params=params)
    spark.sparkContext.setLogLevel("ERROR")

    print(f"✅ Spark NLP Version: {sparknlp.version()}")
    print(f"✅ Spark NLP JSL Version: {sparknlp_jsl.version()}")
    print("✅ Spark session initialized successfully")

except Exception as e:
    print(f"❌ Error starting Spark session: {e}")
    raise

spark


🚀 GPU Detected: NVIDIA L4
✅ GPU acceleration enabled in Spark configuration
Starting Spark session...
✅ Spark NLP Version: 6.1.3
✅ Spark NLP JSL Version: 6.1.1
✅ Spark session initialized successfully


## 3. Dataset Loading


In [ ]:
# Download dataset
import requests

data_dir = Path(f"{PROJECT_FOLDER}/data/raw")
data_dir.mkdir(parents=True, exist_ok=True)

url = "https://raw.githubusercontent.com/JohnSnowLabs/spark-nlp-workshop/master/tutorials/Certification_Trainings/Healthcare/data/mtsamples_classifier.csv"
file_path = data_dir / "mtsamples_classifier.csv"

if not file_path.exists():
    print(f"Downloading mtsamples_classifier dataset from {url}...")
    response = requests.get(url, timeout=60)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)
    print(f"Dataset saved to {file_path}")
else:
    print(f"Dataset already exists at {file_path}")

df = pd.read_csv(file_path)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()


Dataset already exists at /content/drive/MyDrive/john_snow_labs_ner/data/raw/mtsamples_classifier.csv
Dataset shape: (638, 2)
Columns: ['category', 'text']


,category,text
0,Gastroenterology,PROCEDURES PERFORMED: Colonoscopy. INDICATION...
1,Gastroenterology,OPERATION 1. Ivor-Lewis esophagogastrectomy. ...
2,Gastroenterology,PREOPERATIVE DIAGNOSES: 1. Gastroesophageal r...
3,Gastroenterology,PROCEDURE: Colonoscopy. PREOPERATIVE DIAGNOSE...
4,Gastroenterology,PREOPERATIVE DIAGNOSIS: Right colon tumor. PO...


In [ ]:
# Prepare text dataframe
text_df = df.copy()

# Create text_id if not exists
if 'text_id' not in text_df.columns:
    text_df['text_id'] = range(len(text_df))

# Ensure text column exists
if 'text' not in text_df.columns:
    # Try to find text column
    text_cols = [col for col in text_df.columns if 'text' in col.lower() or 'description' in col.lower()]
    if text_cols:
        text_df['text'] = text_df[text_cols[0]]
    else:
        raise ValueError("No text column found in dataset")

# Select and rename columns
text_df = text_df[['text_id', 'text']].copy()

# Save to Google Drive
output_path = Path(f"{PROJECT_FOLDER}/data/processed/text_data.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
text_df.to_csv(output_path, index=False)

print(f"✅ Saved {len(text_df)} texts to {output_path}")
text_df.head()


✅ Saved 638 texts to /content/drive/MyDrive/john_snow_labs_ner/data/processed/text_data.csv


,text_id,text
0,0,PROCEDURES PERFORMED: Colonoscopy. INDICATION...
1,1,OPERATION 1. Ivor-Lewis esophagogastrectomy. ...
2,2,PREOPERATIVE DIAGNOSES: 1. Gastroesophageal r...
3,3,PROCEDURE: Colonoscopy. PREOPERATIVE DIAGNOSE...
4,4,PREOPERATIVE DIAGNOSIS: Right colon tumor. PO...


## 4. NER Pipeline Execution


In [ ]:
"""
NER Pipeline Module
Creates and executes Spark NLP Healthcare NER pipeline with multiple models
"""

import sparknlp
import sparknlp_jsl
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetector, Tokenizer, WordEmbeddingsModel
from sparknlp_jsl.annotator import (
    MedicalNerModel,
    NerConverter
)
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from typing import Optional, Dict, List
import warnings
warnings.filterwarnings('ignore')


class NERPipeline:
    """NER Pipeline with multiple pre-trained models"""

    def __init__(self, spark: SparkSession, license_secret: Optional[str] = None):
        """
        Initialize NER Pipeline

        Args:
            spark: SparkSession instance
            license_secret: Spark NLP Healthcare license secret (if not already configured)
        """
        self.spark = spark
        self.license_secret = license_secret
        self.pipeline = None
        self.models = {}

    def create_pipeline(self, prioritize_posology_deid: bool = True):
        """
        Create NER pipeline with multiple models

        Args:
            prioritize_posology_deid: If True, posology and deid models take priority
        """
        # Document Assembler
        document_assembler = DocumentAssembler()\
            .setInputCol("text")\
            .setOutputCol("document")\
            .setCleanupMode("shrink")

        # Sentence Detector
        sentence_detector = SentenceDetector()\
            .setInputCols(["document"])\
            .setOutputCol("sentence")\
            .setExplodeSentences(True)

        # Tokenizer
        tokenizer = Tokenizer()\
            .setInputCols(["sentence"])\
            .setOutputCol("token")

        # Word Embeddings (required for MedicalNerModel)
        # Clinical embeddings are used for all NER models
        print("Loading clinical word embeddings...")
        word_embeddings = WordEmbeddingsModel.pretrained("embeddings_clinical", "en", "clinical/models")\
            .setInputCols(["sentence", "token"])\
            .setOutputCol("embeddings")
        print("✅ Clinical embeddings loaded")

        # NER Models
        # Note: MedicalNerModel requires 3 inputs: document, token, and word_embeddings
        # 1. Clinical NER Model
        print("Loading ner_clinical model...")
        ner_clinical = MedicalNerModel.pretrained("ner_clinical", "en", "clinical/models")\
            .setInputCols(["sentence", "token", "embeddings"])\
            .setOutputCol("ner_clinical")

        # 2. DeID Generic Augmented Model
        print("Loading ner_deid_generic_augmented model...")
        ner_deid = MedicalNerModel.pretrained("ner_deid_generic_augmented", "en", "clinical/models")\
            .setInputCols(["sentence", "token", "embeddings"])\
            .setOutputCol("ner_deid")

        # 3. Posology Model (for Drug and Dosage)
        print("Loading ner_posology model...")
        ner_posology = MedicalNerModel.pretrained("ner_posology", "en", "clinical/models")\
            .setInputCols(["sentence", "token", "embeddings"])\
            .setOutputCol("ner_posology")

        # Store models
        self.models = {
            'clinical': ner_clinical,
            'deid': ner_deid,
            'posology': ner_posology
        }

        # Create pipeline stages
        # Note: word_embeddings must come before NER models
        stages = [
            document_assembler,
            sentence_detector,
            tokenizer,
            word_embeddings,  # Required for MedicalNerModel
            ner_clinical,
            ner_deid,
            ner_posology
        ]

        # If prioritizing, we need to merge results
        # For now, we'll run all models and merge in post-processing
        if prioritize_posology_deid:
            # Add NerConverter for each model
            ner_converter_clinical = NerConverter()\
                .setInputCols(["document", "token", "ner_clinical"])\
                .setOutputCol("chunk_clinical")

            ner_converter_deid = NerConverter()\
                .setInputCols(["document", "token", "ner_deid"])\
                .setOutputCol("chunk_deid")

            ner_converter_posology = NerConverter()\
                .setInputCols(["document", "token", "ner_posology"])\
                .setOutputCol("chunk_posology")

            stages.extend([
                ner_converter_clinical,
                ner_converter_deid,
                ner_converter_posology
            ])

        self.pipeline = Pipeline(stages=stages)
        return self.pipeline

    def fit_transform(self, data):
        """
        Fit and transform data through pipeline

        Args:
            data: Spark DataFrame with 'text' column

        Returns:
            Transformed DataFrame with NER results
        """
        if self.pipeline is None:
            raise ValueError("Pipeline not created. Call create_pipeline() first.")

        model = self.pipeline.fit(data)
        result = model.transform(data)
        return result

    def filter_posology_entities(self, ner_result, keep_entities: List[str] = ["Drug", "Dosage"]):
        """
        Filter posology entities to keep only Drug and Dosage

        Args:
            ner_result: NER result from posology model
            keep_entities: List of entity types to keep

        Returns:
            Filtered NER results
        """
        # This would be implemented based on the actual structure of NER results
        # For now, this is a placeholder
        return ner_result

    def merge_ner_results(self, result_df, prioritize_posology_deid: bool = True):
        """
        Merge results from multiple NER models with priority

        Priority order (if prioritize_posology_deid=True):
        1. Posology (Drug, Dosage)
        2. DeID (PHI entities)
        3. Clinical (other clinical entities)

        Args:
            result_df: DataFrame with NER results from all models
            prioritize_posology_deid: Whether to prioritize posology and deid

        Returns:
            DataFrame with merged NER results
        """
        # This is a complex operation that requires:
        # 1. Extracting entities from each model
        # 2. Resolving conflicts based on priority
        # 3. Creating a unified entity list

        # For now, return the original dataframe
        # Full implementation would merge chunks with priority logic
        return result_df

    def extract_entities(self, result_df) -> List[Dict]:
        """
        Extract entities from pipeline results

        Args:
            result_df: DataFrame with NER results

        Returns:
            List of entity dictionaries with text_id, begin, end, chunk, entity
        """
        entities = []

        # Extract entities from each model
        # This is a simplified version - actual implementation would need
        # to handle the Spark DataFrame structure properly

        return entities



In [ ]:
# Create NER pipeline
ner_pipeline = NERPipeline(spark)
pipeline = ner_pipeline.create_pipeline(prioritize_posology_deid=True)

print("✅ NER pipeline created")
print("Models in pipeline:")
print("  - ner_clinical")
print("  - ner_deid_generic_augmented")
print("  - ner_posology")

# Check GPU status for inference
try:
    import torch
    if torch.cuda.is_available():
        print(f"\n🚀 GPU available for inference: {torch.cuda.get_device_name(0)}")
        print("   Model inference will use GPU acceleration")
    else:
        print("\n⚠️  No GPU detected - using CPU for inference")
        print("   Enable GPU for faster inference: Runtime → Change runtime type → GPU")
except:
    print("\n⚠️  GPU check unavailable")

Loading clinical word embeddings...
embeddings_clinical download started this may take some time.
Approximate size to download 1.6 GB
[OK!]
✅ Clinical embeddings loaded
Loading ner_clinical model...
ner_clinical download started this may take some time.
Approximate size to download 13.9 MB
[OK!]
Loading ner_deid_generic_augmented model...
ner_deid_generic_augmented download started this may take some time.
Approximate size to download 13.8 MB
[OK!]
Loading ner_posology model...
ner_posology download started this may take some time.
Approximate size to download 13.8 MB
[OK!]
✅ NER pipeline created
Models in pipeline:
  - ner_clinical
  - ner_deid_generic_augmented
  - ner_posology

🚀 GPU available for inference: NVIDIA L4
   Model inference will use GPU acceleration


In [ ]:
# Convert pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(text_df)
print(f"Spark DataFrame created with {spark_df.count()} rows")
spark_df.show(5, truncate=100)

Spark DataFrame created with 638 rows
+-------+----------------------------------------------------------------------------------------------------+
|text_id|                                                                                                text|
+-------+----------------------------------------------------------------------------------------------------+
|      0| PROCEDURES PERFORMED: Colonoscopy. INDICATIONS: Renewed symptoms likely consistent with active f...|
|      1| OPERATION 1. Ivor-Lewis esophagogastrectomy. 2. Feeding jejunostomy. 3. Placement of two right-s...|
|      2| PREOPERATIVE DIAGNOSES: 1. Gastroesophageal reflux disease. 2. Chronic dyspepsia. POSTOPERATIVE ...|
|      3| PROCEDURE: Colonoscopy. PREOPERATIVE DIAGNOSES: Rectal bleeding and perirectal abscess. POSTOPER...|
|      4| PREOPERATIVE DIAGNOSIS: Right colon tumor. POSTOPERATIVE DIAGNOSES: 1. Right colon cancer. 2. As...|
+-------+-----------------------------------------------------------------

In [ ]:
# Run NER pipeline
print("Running NER pipeline... This may take several minutes...")
result_df = ner_pipeline.fit_transform(spark_df)

print("✅ NER pipeline completed")
result_df.select("text_id", "text", "chunk_clinical", "chunk_deid", "chunk_posology").show(5, truncate=100)

Running NER pipeline... This may take several minutes...
✅ NER pipeline completed
+-------+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+----------+----------------------------------------------------------------------------------------------------+
|text_id|                                                                                                text|                                                                                      chunk_clinical|chunk_deid|                                                                                      chunk_posology|
+-------+----------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------+----------+------------------------------------------------

## 5. Entity Extraction and Merging


In [ ]:
# Extract entities from NER results with priority merging
# Priority: Posology > DeID > Clinical

def extract_entities_from_ner_results(result_df, text_df):
    """
    Extract entities from NER pipeline results and create entity dataframe.
    Priority: Posology > DeID > Clinical

    Args:
        result_df: Spark DataFrame with NER results
        text_df: Pandas DataFrame with text data

    Returns:
        Pandas DataFrame with merged entities
    """
    entities_list = []

    # Collect clinical entities
    clinical_results = result_df.select(
        "text_id",
        F.explode(F.arrays_zip(
            result_df["chunk_clinical"].result,
            result_df["chunk_clinical"].begin,
            result_df["chunk_clinical"].end,
            result_df["chunk_clinical"].metadata
        )).alias("clinical_chunk")
    ).select(
        "text_id",
        F.expr("clinical_chunk['0']").alias("chunk"),
        F.expr("clinical_chunk['1']").alias("begin"),
        F.expr("clinical_chunk['2']").alias("end"),
        F.expr("clinical_chunk['3']['entity']").alias("entity")
    ).collect()

    # Process clinical entities
    clinical_entities = {}
    for row in clinical_results:
        text_id = row.text_id
        if text_id not in clinical_entities:
            clinical_entities[text_id] = []
        clinical_entities[text_id].append({
            'begin': row.begin,
            'end': row.end,
            'chunk': row.chunk,
            'entity': row.entity,
            'source': 'clinical'
        })

    # Process DeID entities
    deid_results = result_df.select(
        "text_id",
        F.explode(F.arrays_zip(
            result_df["chunk_deid"].result,
            result_df["chunk_deid"].begin,
            result_df["chunk_deid"].end,
            result_df["chunk_deid"].metadata
        )).alias("deid_chunk")
    ).select(
        "text_id",
        F.expr("deid_chunk['0']").alias("chunk"),
        F.expr("deid_chunk['1']").alias("begin"),
        F.expr("deid_chunk['2']").alias("end"),
        F.expr("deid_chunk['3']['entity']").alias("entity")
    ).collect()

    deid_entities = {}
    for row in deid_results:
        text_id = row.text_id
        if text_id not in deid_entities:
            deid_entities[text_id] = []
        deid_entities[text_id].append({
            'begin': row.begin,
            'end': row.end,
            'chunk': row.chunk,
            'entity': row.entity,
            'source': 'deid'
        })

    # Process Posology entities (Drug, Dosage only)
    posology_results = result_df.select(
        "text_id",
        F.explode(F.arrays_zip(
            result_df["chunk_posology"].result,
            result_df["chunk_posology"].begin,
            result_df["chunk_posology"].end,
            result_df["chunk_posology"].metadata
        )).alias("posology_chunk")
    ).select(
        "text_id",
        F.expr("posology_chunk['0']").alias("chunk"),
        F.expr("posology_chunk['1']").alias("begin"),
        F.expr("posology_chunk['2']").alias("end"),
        F.expr("posology_chunk['3']['entity']").alias("entity")
    ).filter(
        F.col("entity").isin(["Drug", "Dosage"])
    ).collect()

    posology_entities = {}
    for row in posology_results:
        text_id = row.text_id
        if text_id not in posology_entities:
            posology_entities[text_id] = []
        posology_entities[text_id].append({
            'begin': row.begin,
            'end': row.end,
            'chunk': row.chunk,
            'entity': row.entity,
            'source': 'posology'
        })

    # Merge entities with priority: Posology > DeID > Clinical
    all_entities = []
    for text_id in text_df['text_id'].values:
        merged = {}

        # Add posology entities (highest priority)
        if text_id in posology_entities:
            for ent in posology_entities[text_id]:
                key = (ent['begin'], ent['end'])
                merged[key] = ent

        # Add DeID entities (medium priority)
        if text_id in deid_entities:
            for ent in deid_entities[text_id]:
                key = (ent['begin'], ent['end'])
                if key not in merged:  # Don't override posology
                    merged[key] = ent

        # Add clinical entities (lowest priority)
        if text_id in clinical_entities:
            for ent in clinical_entities[text_id]:
                key = (ent['begin'], ent['end'])
                if key not in merged:  # Don't override posology or deid
                    merged[key] = ent

        # Convert to list
        for ent in merged.values():
            all_entities.append({
                'text_id': text_id,
                'begin': ent['begin'],
                'end': ent['end'],
                'chunk': ent['chunk'],
                'entity': ent['entity']
            })

    entity_df = pd.DataFrame(all_entities)
    return entity_df


# Extract entities
print("Extracting entities from NER results...")
entity_df = extract_entities_from_ner_results(result_df, text_df)
print(f"✅ Extracted {len(entity_df)} entities")
print(f"\nEntity types: {sorted(entity_df['entity'].unique())}")
print(f"\nEntity distribution:")
print(entity_df['entity'].value_counts())

# Save entities to Google Drive
entity_output_path = Path(f"{PROJECT_FOLDER}/data/processed/entities.csv")
entity_output_path.parent.mkdir(parents=True, exist_ok=True)
entity_df.to_csv(entity_output_path, index=False)
print(f"\n✅ Saved entities to {entity_output_path}")


entity_df.head(10)

In [ ]:
import pandas as pd
from pathlib import Path

# Dosya yolu
entity_output_path = Path(f"{PROJECT_FOLDER}/data/processed/entities.csv")

# CSV dosyasını oku
entity_df = pd.read_csv(entity_output_path)

# İlk 10 satırı göster
entity_df.head(10)


,text_id,begin,end,chunk,entity
0,0,22,32,Colonoscopy,TEST
1,0,48,63,Renewed symptoms,PROBLEM
2,0,104,129,Inflammatory Bowel Disease,PROBLEM
3,0,150,169,conventional therapy,TREATMENT
4,0,181,193,sulfasalazine,TREATMENT
5,0,196,204,cortisone,TREATMENT
6,0,207,219,local therapy,TREATMENT
7,0,272,284,the procedure,TREATMENT
8,0,362,369,bleeding,PROBLEM
9,0,372,380,infection,PROBLEM


In [ ]:
import re

def fix_entity_offsets(text, entity_df):
    corrected = []

    for idx, row in entity_df.iterrows():
        begin = row["begin"]
        end = row["end"]
        chunk = row["chunk"]

        # 1. Substringi çek
        real = text[begin:end]

        # 2. Baştaki boşlukları kaldır
        while real.startswith(" ") and begin < end:
            begin += 1
            real = text[begin:end]

        # 3. Eğer chunk real ile birebir başlamıyorsa → kelime tam değil
        # Regex ile en yakın eşleşmeyi bul
        if chunk.startswith(real) and len(real) < len(chunk):
            # sadece eksik harfi tamamla
            diff = len(chunk) - len(real)
            end += diff
            real = text[begin:end]

        # 4. Eğer hala eşleşmiyorsa → chunk’ı text içinde arayıp gerçek pozisyonu bul
        if chunk not in real:
            # Metinde chunk geçen ilk yer
            m = re.search(re.escape(chunk), text)
            if m:
                begin = m.start()
                end = m.end()

        corrected.append([row["text_id"], begin, end, chunk, row["entity"]])

    return pd.DataFrame(corrected, columns=entity_df.columns)


entity_df = fix_entity_offsets(text_df.loc[text_df['text_id']==0,'text'].values[0], entity_df)
entity_df

,text_id,begin,end,chunk,entity
0,0,23,34,Colonoscopy,TEST
1,0,49,65,Renewed symptoms,PROBLEM
2,0,105,131,Inflammatory Bowel Disease,PROBLEM
3,0,151,171,conventional therapy,TREATMENT
4,0,182,195,sulfasalazine,TREATMENT
...,...,...,...,...,...
39578,637,971,985,any side effects,PROBLEM
39579,637,992,1005,her medication,TREATMENT
39580,637,1044,1050,Ditropan,TREATMENT
39581,637,1067,1077,Sanctura XR,TREATMENT


In [ ]:
result_df.printSchema()

root
 |-- text_id: long (nullable = true)
 |-- text: string (nullable = true)
 |-- document: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- annotatorType: string (nullable = true)
 |    |    |-- begin: integer (nullable = false)
 |    |    |-- end: integer (nullable = false)
 |    |    |-- result: string (nullable = true)
 |    |    |-- metadata: map (nullable = true)
 |    |    |    |-- key: string
 |    |    |    |-- value: string (valueContainsNull = true)
 |    |    |-- embeddings: array (nullable = true)
 |    |    |    |-- element: float (containsNull = false)
 |-- sentence: array (nullable = false)
 |    |-- element: struct (containsNull = true)
 |    |    |-- annotatorType: string (nullable = true)
 |    |    |-- begin: integer (nullable = false)
 |    |    |-- end: integer (nullable = false)
 |    |    |-- result: string (nullable = true)
 |    |    |-- metadata: map (nullable = true)
 |    |    |    |-- key: string
 |    |    |    |-- v

## 6. CoNLL Format Conversion


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetector, Tokenizer
from pyspark.ml import Pipeline
from tqdm import tqdm
from collections import Counter
import pandas as pd
from pathlib import Path
import os


from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import SentenceDetector, Tokenizer
from pyspark.ml import Pipeline
from tqdm import tqdm
from collections import Counter
import pandas as pd
from pathlib import Path
import os


def make_conll(
    text: pd.DataFrame,
    entity: pd.DataFrame,
    project_folder: str,
    save_tag: bool = True,
    save_conll: bool = True,
    verbose: bool = None,
    begin_deviation: int = 0,
    end_deviation: int = 0,
) -> str:

    df_text = text.iloc[:, [0, 1]]
#     df_text = (
#     result_df
#     .select("text_id", "text")
#     .limit(5000)
#     .toPandas()
# ) # limit 5k
    print(len(df_text))
    df_entity = entity.iloc[:, [0, 1, 2, 3, 4]]
    df_text.columns = ["text_id", "text"]
    df_entity.columns = ["text_id", "begin", "end", "chunk", "entity"]
    entity_list = list(df_entity.entity.unique())

    ########--------------1.tag transformation function------------########

    def transform_text(text, entities, verbose=None):

        tag_list = []
        for entity in entities.iterrows():

            begin = entity[1][1] + begin_deviation
            end = entity[1][2] + end_deviation
            chunk = entity[1][3]
            tag = entity[1][4]
            text = text[:end] + f" </END_NER:{tag}> " + text[end:]
            text = text[:begin] + f" <START_NER:{tag}> " + text[begin:]
            tag_list.append(tag)

        sum_of_added_entity = Counter(tag_list)
        sum_of_entity = Counter(entities["entity"].values)

        if verbose:
            print(f"Processed text id   : {entities.text_id.values[:1]}")
            print(
                f"Original Entities   : {sum_of_entity}\nAdded Entities      : {sum_of_added_entity}"
            )
            print(f"Number Equality     : {sum_of_added_entity == sum_of_entity}")
            print("==" * 40)

        if not sum_of_entity == sum_of_added_entity:
            print("There is a problem in text id:")
            print(entities.text_id.values[0])
            raise Exception("Check this text!")

        return text

    ######---------------2.apply_transform_text function ----------------#######

    def apply_tag_ner(df_text, df_entity, save=None, verbose=None):

        for text_id in tqdm(df_text.text_id):
            text = df_text.loc[df_text["text_id"] == text_id]["text"].values[0]
            entities = df_entity.loc[(df_entity["text_id"] == text_id)].sort_values(
                by="begin", ascending=False
            )

            df_text.loc[df_text["text_id"] == text_id, "text"] = transform_text(
                text, entities, verbose=verbose
            )

        if save:
            df_text.to_csv("text_with_ner_tag.csv", index=False, encoding="utf8")

        return df_text

    ##########----------------3.RUNNING TAG FUNCTION---------------#############

    print("Text tagging starting. Applying entities to whole text...\n")
    df = apply_tag_ner(df_text, df_entity, save=save_tag, verbose=verbose)

    ###########---------------4.Spark Pipeline-----------------------###########

    def spark_pipeline(df):
        spark_df = spark.createDataFrame(df)

        documentAssembler = (
            DocumentAssembler()
            .setInputCol("text")
            .setOutputCol("document")
            .setCleanupMode("shrink")
        )

        sentenceDetector = (
            SentenceDetector()
            .setInputCols(["document"])
            .setOutputCol("sentences")
            .setExplodeSentences(True)
        )

        tokenizer = Tokenizer().setInputCols(["sentences"]).setOutputCol("token")

        nlpPipeline = Pipeline(stages=[documentAssembler, sentenceDetector, tokenizer])

        empty_df = spark.createDataFrame([[""]]).toDF("text")
        pipelineModel = nlpPipeline.fit(empty_df)

        result = pipelineModel.transform(spark_df.select(["text"]))

        return result.select("token.result").toPandas()

    print("\n\nSpark pipeline is running...")

    df_final = spark_pipeline(df)

    #########--------------5.CoNLL Function--------------------#############

    def build_conll(df_final, tag_list, save=None):

        header = "-DOCSTART- -X- -X- O\n\n"
        conll_text = ""
        chunks = []
        tag_list = tag_list
        tag = "O"  # token tag
        ct = "B"  # chunk tag part B or I

        for sentence_tokens in tqdm(df_final.result[:]):
            for token in sentence_tokens:
                if token.startswith("<START_NER:"):
                    tag = token.split(":")[1][:-1]
                    if tag not in tag_list:
                        tag = "O"
                        conll_text += f"{token} NN NN {tag}\n"

                    continue

                if token.startswith("</END_NER:") and tag != "O":
                    for i, chunk in enumerate(chunks):
                        ct = "B" if i == 0 else "I"
                        conll_text += f"{chunk} NNP NNP {ct}-{tag}\n"

                    chunks = []
                    tag = "O"
                    continue

                if tag != "O":
                    chunks.append(token)
                    continue

                if tag == "O":
                    conll_text += f"{token} NN NN {tag}\n"
                    continue

            conll_text += "\n"

        if save:
            output_path = Path(project_folder) / "data/conll/conll2003_text_file.conll"
            output_path.parent.mkdir(parents=True, exist_ok=True)
            with open(output_path, "w+", encoding="utf8") as f:
                f.write(header)
                f.write(conll_text)
            print(f"✅ CoNLL file saved to {output_path}")

        print("\nDONE!")
        return conll_text

    ########----------------6.RUNNING CONLL FUNCTION--------------------########

    print("Conll file is being created...\n")
    return build_conll(df_final, tag_list=entity_list, save=save_conll)



In [ ]:
import sys, pkgutil, subprocess
if pkgutil.find_loader("tqdm") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tqdm"])

from tqdm.auto import tqdm
from collections import Counter

In [ ]:
conll_text = make_conll(
    text=text_df,
    entity=entity_df,
    project_folder=PROJECT_FOLDER,
    save_tag=True,
    save_conll=True,
    verbose=False
)


638
Text tagging starting. Applying entities to whole text...



  0%|          | 0/638 [00:00<?, ?it/s]



Spark pipeline is running...
Conll file is being created...



  0%|          | 0/26012 [00:00<?, ?it/s]

✅ CoNLL file saved to /content/drive/MyDrive/john_snow_labs_ner/data/conll/conll2003_text_file.conll

DONE!


In [ ]:
# Checking conll string
print(conll_text[:560])

PROCEDURES NN NN O
PERFORMED NN NN O
: NN NN O
Colonoscopy NNP NNP B-TEST
. NN NN O

INDICATIONS NN NN O
: NN NN O
Renewed NNP NNP B-PROBLEM
symptoms NNP NNP I-PROBLEM
likely NN NN O
consistent NN NN O
with NN NN O
active NN NN O
flare NN NN O
of NN NN O
Inflammatory NNP NNP B-PROBLEM
Bowel NNP NNP I-PROBLEM
Disease NNP NNP I-PROBLEM
, NN NN O
not NN NN O
responsive NN NN O
to NN NN O
conventional NNP NNP B-TREATMENT
therapy NNP NNP I-TREATMENT
including NN NN O
sulfasalazine NNP NNP B-TREATMENT
, NN NN O
cortisone NNP NNP B-TREATMENT
, NN NN O
local NNP


## 7. CoNLL File Validation

Verify that the CoNLL file was created correctly and can be read by Spark NLP

In [ ]:
# Read and display sample CoNLL content
conll_file_path = Path(f"{PROJECT_FOLDER}/data/conll/conll2003_text_file.conll")

print("📄 CoNLL File Sample (First 100 lines):")
print("=" * 80)

with open(conll_file_path, 'r', encoding='utf8') as f:
    lines = f.readlines()
    for i, line in enumerate(lines[:100], 1):
        print(f"{i:3d}: {line}", end='')

print("\n" + "=" * 80)
print(f"\n📊 CoNLL File Statistics:")
print(f"  Total lines: {len(lines)}")
print(f"  File size: {conll_file_path.stat().st_size / 1024:.2f} KB")
print(f"  File location: {conll_file_path}")

📄 CoNLL File Sample (First 100 lines):
  1: -DOCSTART- -X- -X- O
  2: 
  3: PROCEDURES NN NN O
  4: PERFORMED NN NN O
  5: : NN NN O
  6: Colonoscopy NNP NNP B-TEST
  7: . NN NN O
  8: 
  9: INDICATIONS NN NN O
 10: : NN NN O
 11: Renewed NNP NNP B-PROBLEM
 12: symptoms NNP NNP I-PROBLEM
 13: likely NN NN O
 14: consistent NN NN O
 15: with NN NN O
 16: active NN NN O
 17: flare NN NN O
 18: of NN NN O
 19: Inflammatory NNP NNP B-PROBLEM
 20: Bowel NNP NNP I-PROBLEM
 21: Disease NNP NNP I-PROBLEM
 22: , NN NN O
 23: not NN NN O
 24: responsive NN NN O
 25: to NN NN O
 26: conventional NNP NNP B-TREATMENT
 27: therapy NNP NNP I-TREATMENT
 28: including NN NN O
 29: sulfasalazine NNP NNP B-TREATMENT
 30: , NN NN O
 31: cortisone NNP NNP B-TREATMENT
 32: , NN NN O
 33: local NNP NNP B-TREATMENT
 34: therapy NNP NNP I-TREATMENT
 35: . NN NN O
 36: 
 37: PROCEDURE NN NN O
 38: : NN NN O
 39: Informed NN NN O
 40: consent NN NN O
 41: was NN NN O
 42: obtained NN NN O
 43: prior NN NN O
 44:

## 8. NER Visualization with Spark NLP Display

Visualize the NER results using the spark-nlp-display library

In [ ]:
# Install spark-nlp-display if not already installed
try:
    import sparknlp_display
    print("✅ spark-nlp-display already installed")
except ImportError:
    print("Installing spark-nlp-display...")
    %pip install -q spark-nlp-display
    import sparknlp_display
    print("✅ spark-nlp-display installed successfully")

from sparknlp_display import NerVisualizer

✅ spark-nlp-display already installed


In [ ]:
# Visualize NER results for a sample of texts
# Select a few examples to visualize
sample_size = 5
sample_texts = result_df.select("text_id", "text", "chunk_clinical", "chunk_deid", "chunk_posology")\
    .limit(sample_size)\
    .collect()

print(f"🎨 Visualizing NER results for {sample_size} sample texts...\n")

# For each sample, create visualizations for each model
for idx, row in enumerate(sample_texts, 1):
    print(f"\n{'='*100}")
    print(f"📝 Sample {idx}/{sample_size} - Text ID: {row.text_id}")
    print(f"{'='*100}")
    print(f"\nOriginal Text (first 200 chars):")
    print(f"{row.text[:200]}...")
    print(f"\n{'-'*100}")

🎨 Visualizing NER results for 5 sample texts...


📝 Sample 1/5 - Text ID: 0

Original Text (first 200 chars):
 PROCEDURES PERFORMED: Colonoscopy. INDICATIONS: Renewed symptoms likely consistent with active flare of Inflammatory Bowel Disease, not responsive to conventional therapy including sulfasalazine, cor...

----------------------------------------------------------------------------------------------------

📝 Sample 2/5 - Text ID: 0

Original Text (first 200 chars):
 PROCEDURES PERFORMED: Colonoscopy. INDICATIONS: Renewed symptoms likely consistent with active flare of Inflammatory Bowel Disease, not responsive to conventional therapy including sulfasalazine, cor...

----------------------------------------------------------------------------------------------------

📝 Sample 3/5 - Text ID: 0

Original Text (first 200 chars):
 PROCEDURES PERFORMED: Colonoscopy. INDICATIONS: Renewed symptoms likely consistent with active flare of Inflammatory Bowel Disease, not responsive to conve

In [ ]:
# Display sample NER results in a readable format
print("\n🎨 Sample NER Results Visualization")
print("="*100)

# Select a few samples to display
sample_count = 3
samples = result_df.limit(sample_count).collect()

for idx, sample in enumerate(samples, 1):
    print(f"\n{'='*100}")
    print(f"📝 Sample {idx}/{sample_count} - Text ID: {sample.text_id}")
    print(f"{'='*100}")

    # Display original text
    print(f"\n📄 Original Text:")
    print(f"{sample.text[:300]}...")

    # Display Clinical entities
    print(f"\n🔬 Clinical Entities (ner_clinical):")
    if sample.chunk_clinical:
        for i, chunk in enumerate(sample.chunk_clinical[:10]):  # Show first 10
            entity_type = chunk.metadata.get('entity', 'Unknown')
            print(f"  {i+1}. [{entity_type}] {chunk.result}")
        if len(sample.chunk_clinical) > 10:
            print(f"  ... and {len(sample.chunk_clinical) - 10} more")
    else:
        print("  No entities found")

    # Display DeID entities
    print(f"\n🔒 De-Identification Entities (ner_deid):")
    if sample.chunk_deid:
        for i, chunk in enumerate(sample.chunk_deid[:10]):  # Show first 10
            entity_type = chunk.metadata.get('entity', 'Unknown')
            print(f"  {i+1}. [{entity_type}] {chunk.result}")
        if len(sample.chunk_deid) > 10:
            print(f"  ... and {len(sample.chunk_deid) - 10} more")
    else:
        print("  No entities found")

    # Display Posology entities
    print(f"\n💊 Posology Entities - Drug & Dosage (ner_posology):")
    if sample.chunk_posology:
        for i, chunk in enumerate(sample.chunk_posology[:10]):  # Show first 10
            entity_type = chunk.metadata.get('entity', 'Unknown')
            print(f"  {i+1}. [{entity_type}] {chunk.result}")
        if len(sample.chunk_posology) > 10:
            print(f"  ... and {len(sample.chunk_posology) - 10} more")
    else:
        print("  No entities found")

print(f"\n{'='*100}\n")


🎨 Sample NER Results Visualization

📝 Sample 1/3 - Text ID: 0

📄 Original Text:
 PROCEDURES PERFORMED: Colonoscopy. INDICATIONS: Renewed symptoms likely consistent with active flare of Inflammatory Bowel Disease, not responsive to conventional therapy including sulfasalazine, cortisone, local therapy. PROCEDURE: Informed consent was obtained prior to the procedure with special ...

🔬 Clinical Entities (ner_clinical):
  1. [TEST] Colonoscopy

🔒 De-Identification Entities (ner_deid):
  No entities found

💊 Posology Entities - Drug & Dosage (ner_posology):
  No entities found

📝 Sample 2/3 - Text ID: 0

📄 Original Text:
 PROCEDURES PERFORMED: Colonoscopy. INDICATIONS: Renewed symptoms likely consistent with active flare of Inflammatory Bowel Disease, not responsive to conventional therapy including sulfasalazine, cortisone, local therapy. PROCEDURE: Informed consent was obtained prior to the procedure with special ...

🔬 Clinical Entities (ner_clinical):
  1. [PROBLEM] Renewed symptoms
 

In [ ]:
print(result_df.columns)


['text_id', 'text', 'document', 'sentence', 'token', 'embeddings', 'ner_clinical', 'ner_deid', 'ner_posology', 'chunk_clinical', 'chunk_deid', 'chunk_posology']


In [ ]:
from sparknlp_display import NerVisualizer

ner_vis = NerVisualizer()

sample_results = result_df.limit(3).collect()

print("🎨 Visualizing NER Results:")
print("=" * 80)

for idx, row in enumerate(sample_results, 1):
    print(f"\n📄 Example {idx} - Text ID: {row.text_id}")
    print("-" * 80)

    print("\n🔹 Clinical NER:")
    ner_vis.display(
        result=row,
        label_col='chunk_clinical',
        document_col='document',
        labels=['PROBLEM', 'TREATMENT', 'TEST'],
        return_html=False
    )

    # DeID entities
    print("\n🔹 DeID NER:")
    ner_vis.display(
        result=row,
        label_col='chunk_deid',
        document_col='document',
        return_html=False
    )

    # Posology entities
    print("\n🔹 Posology NER (Drug & Dosage):")
    ner_vis.display(
        result=row,
        label_col='chunk_posology',
        document_col='document',
        return_html=False
    )

    print("=" * 80)

🎨 Visualizing NER Results:

📄 Example 1 - Text ID: 0
--------------------------------------------------------------------------------

🔹 Clinical NER:



🔹 DeID NER:



🔹 Posology NER (Drug & Dosage):



📄 Example 2 - Text ID: 0
--------------------------------------------------------------------------------

🔹 Clinical NER:



🔹 DeID NER:



🔹 Posology NER (Drug & Dosage):



📄 Example 3 - Text ID: 0
--------------------------------------------------------------------------------

🔹 Clinical NER:



🔹 DeID NER:



🔹 Posology NER (Drug & Dosage):


## Summary

✅ **Data preparation completed!**

**Created files in Google Drive:**
- `data/processed/text_data.csv` - Prepared text data
- `data/processed/entities.csv` - Extracted entities
- `data/conll/conll2003_text_file.conll` - CoNLL format training data

**Next step:** Run `training.ipynb` notebook to train the custom NER model.
